<a href="https://colab.research.google.com/github/estefania-apaza/inferencia-causal-proyecto-lengua-enaho2024/blob/main/Actividad%202/TC2_Salinas_Apaza_Perez_Marcos_20231724_20230487_20212237_20221214.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tarea Calificada 2


**Efecto de tener una lengua materna indígena en la confianza hacia la Policía Nacional**

Base de datos: Encuesta Nacional de Hogares 2024

X: Lengua materna indígena

Y: Nivel de confianza en la Policía

Enlace del repositorio de GitHub: https://github.com/estefania-apaza/inferencia-causal-proyecto-lengua-enaho2024/tree/main



*   20231724 - Adrian Salinas Alcántara
*   20230487 - Estefanía Apaza Díaz
*   20212237 - Maria Jesus Perez Zarate
*   20221214 - Fabiana Marcos




# Regresión Lineal Múltiple

In [21]:
import os
import seaborn as sns
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from statsmodels.stats.diagnostic import het_breuschpagan, het_white
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [22]:
# Cargamos la base modificada en la TC1

import pandas as pd

# URL "raw" de tu archivo CSV en GitHub
url = "https://raw.githubusercontent.com/estefania-apaza/inferencia-causal-proyecto-lengua-enaho2024/refs/heads/main/Actividad%202/enaho_limpia_t1.csv"

# Cargar el CSV
data = pd.read_csv(url, encoding="latin-1")

data.head(2)
data.columns

Index(['conglome', 'vivienda', 'hogar', 'codperso', 'p208a', 'p207', 'p300a',
       'p301a', 'p1$06', 'estrsocial', 'dominio', 'dom_2', 'dom_3', 'dom_4',
       'dom_5', 'dom_6', 'dom_7', 'dom_8', 'estrsoc_2.0', 'estrsoc_3.0',
       'estrsoc_4.0', 'estrsoc_5.0', 'estrsoc_6.0', 'educ_2.0', 'educ_3.0',
       'educ_4.0', 'educ_5.0', 'educ_6.0', 'educ_7.0', 'educ_8.0', 'educ_10.0',
       'educ_11.0', 'educ_12.0', 'lengua_indigena', 'conf_policia'],
      dtype='object')

**La base cargada fue preparada para la Tarea Calificada 1**. Se presenta una descripción:

Variable independiente: lengua_indigena

```
0: No habla lengua indígena
1: Habla lengua indígena
```

Variable dependiente: conf_policia

```
# Ya se han eliminado los NN
1. Nada
2. Poco
3. Suficiente
4. Bastante
```

Variables de control:

*   p208a: edad (en números)
*   p207: sexo
```
1. Hombre
2. Mujer
```
*   dom_*
```
# Dummies basadas en los siguientes valores:
1.Costa Norte
2.Costa Centro
3.Costa Sur
4.Sierra Norte
5.Sierra Centro
6.Sierra Sur
7.Selva
8.Lima Metropolitana
```

*   estrsoc_*

```
# Dummies basadas en los siguientes valores:
1.A
2.B
3.C
4.D
5.E
6. Rural
```

*   educ_*

```
# Dummies basadas en la pregunta: ¿Cuál es el último año o grado de estudios y nivel que aprobó?
```

**Sobre esta base se trabajará a continuación.**

In [23]:
# Revisamos que todas las variables sean numéricas
# Solo colocamos dos dummies de cada uno como ejemplo

variables = ['p208a', 'p207','p301a','educ_4.0','educ_5.0','lengua_indigena','conf_policia','dom_5', 'dom_6','estrsoc_2.0', 'estrsoc_3.0']

data[variables].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5046 entries, 0 to 5045
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   p208a            5046 non-null   int64  
 1   p207             5046 non-null   int64  
 2   p301a            5046 non-null   float64
 3   educ_4.0         5046 non-null   int64  
 4   educ_5.0         5046 non-null   int64  
 5   lengua_indigena  5046 non-null   int64  
 6   conf_policia     5046 non-null   float64
 7   dom_5            5046 non-null   int64  
 8   dom_6            5046 non-null   int64  
 9   estrsoc_2.0      5046 non-null   int64  
 10  estrsoc_3.0      5046 non-null   int64  
dtypes: float64(2), int64(9)
memory usage: 433.8 KB


In [24]:
# Evaluamos la mediana de conf_policia para crear la variable dicotómica
mediana = data['conf_policia'].median()
print("Mediana de conf_policia:", mediana)

Mediana de conf_policia: 2.0


In [25]:
# Creamos la variable dicotómica usando la mediana
data['conf_policia_dic'] = (data['conf_policia'] > data['conf_policia'].median()).astype(int)

# Revisamos distribución
data['conf_policia_dic'].value_counts()

,count
conf_policia_dic,
0,4065
1,981


In [26]:
# Cambiamos nombres de variables para facilitar el tratamiento

data = data.rename(columns={
    'p208a': 'edad',
    'p207': 'sexo',})

# Filtramos la data por las columnas a usar

cols_usar = ['edad', 'sexo',
             'dom_2', 'dom_3', 'dom_4', 'dom_5', 'dom_6', 'dom_7', 'dom_8',
             'estrsoc_2.0', 'estrsoc_3.0', 'estrsoc_4.0', 'estrsoc_5.0', 'estrsoc_6.0',
             'educ_2.0', 'educ_3.0', 'educ_4.0', 'educ_5.0', 'educ_6.0', 'educ_7.0', 'educ_8.0', 'educ_10.0', 'educ_11.0', 'educ_12.0',
             'lengua_indigena', 'conf_policia_dic', 'conf_policia']

data = data[cols_usar].copy()

In [27]:
# Revisamos la data final para el análisis
data.head(3)

,edad,sexo,dom_2,dom_3,dom_4,dom_5,dom_6,dom_7,dom_8,estrsoc_2.0,...,educ_5.0,educ_6.0,educ_7.0,educ_8.0,educ_10.0,educ_11.0,educ_12.0,lengua_indigena,conf_policia_dic,conf_policia
0,65,1,0,0,1,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,1.0
1,33,2,0,0,1,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,2.0
2,58,1,0,0,1,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,2.0


## DESDE AQUÍ EDITAR

In [28]:
# ==============================================================================
# MODELO BASE (MCO)
# ==============================================================================
import statsmodels.api as sm
import pandas as pd
from statsmodels.stats.diagnostic import het_white
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Transformación de variables
# Volvemos a crear una variable dummy "mujer" (0=Hombre, 1=Mujer) para facilitar interpretación
data['mujer'] = data['sexo'].map({1: 0, 2: 1})

# Definimos VD
y = data['conf_policia']

# Agrupamos variables de control (omitiremos la categoría base para evitar colinealidad)
controles_educ = ['educ_2.0', 'educ_3.0', 'educ_4.0', 'educ_5.0', 'educ_6.0', 'educ_7.0', 'educ_8.0', 'educ_10.0', 'educ_11.0', 'educ_12.0']
controles_estrato = ['estrsoc_2.0', 'estrsoc_3.0', 'estrsoc_4.0', 'estrsoc_5.0', 'estrsoc_6.0']

# Definimos las VI para cada modelo
X_simple = data[['lengua_indigena']]
X_multiple = data[['lengua_indigena', 'edad', 'mujer'] + controles_educ + controles_estrato]

# Agregaremos la constante en statsmodels
X_simple = sm.add_constant(X_simple)
X_multiple = sm.add_constant(X_multiple)

#  Regresión 1 (Simple)
print("=================== REGRESIÓN 1: MODELO SIMPLE ===================")
modelo_1 = sm.OLS(y, X_simple).fit()
print(modelo_1.summary())

# Regresión 2 (con controles)
print("\n=================== REGRESIÓN 2: MODELO MÚLTIPLE ===================")
modelo_2 = sm.OLS(y, X_multiple).fit()
print(modelo_2.summary())

=================== REGRESIÓN 1: MODELO SIMPLE ===================
                            OLS Regression Results                            
Dep. Variable:           conf_policia   R-squared:                       0.016
Model:                            OLS   Adj. R-squared:                  0.016
Method:                 Least Squares   F-statistic:                     82.29
Date:                Mon, 02 Mar 2026   Prob (F-statistic):           1.65e-19
Time:                        02:26:08   Log-Likelihood:                -6181.3
No. Observations:                5046   AIC:                         1.237e+04
Df Residuals:                    5044   BIC:                         1.238e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------

In [29]:
# ==============================================================================
# Diagnósticos (Sobre modelo múltiple) Y selección
# ==============================================================================

# Test de multicolinealidad (VIF)
print("--- Factor de inflación de varianza (VIF) ---")
vif_data = pd.DataFrame()
vif_data["Variable"] = X_multiple.columns
# Calculamos el VIF iterando por cada columna
vif_data["VIF"] = [variance_inflation_factor(X_multiple.values, i) for i in range(X_multiple.shape[1])]
print(vif_data)
# Si VIF > 10 indica problemas de multicolinealidad

# Test de heterocedasticidad (White)
print("\n--- Prueba de White de Heterocedasticidad ---")
residuos = modelo_2.resid
white_test = het_white(residuos, X_multiple)
print(f"Estadístico LM: {white_test[0]:.4f}")
print(f"p-value LM: {white_test[1]:.4f}")
# Si el p-value es menor a 0.05, existe heterocedasticidad en el modelo

# C. Comparación de criterios (selección)
print("\n--- SELECCIÓN DE MODELOS (Criterios AIC y BIC) ---")
print("Regla: Valores más bajos indican un mejor modelo.")
print(f"Modelo 1 (Simple)   -> AIC: {modelo_1.aic:.2f}  |  BIC: {modelo_1.bic:.2f}")
print(f"Modelo 2 (Múltiple) -> AIC: {modelo_2.aic:.2f}  |  BIC: {modelo_2.bic:.2f}")

--- Factor de inflación de varianza (VIF) ---
           Variable         VIF
0             const  124.037861
1   lengua_indigena    1.159673
2              edad    1.352906
3             mujer    1.052000
4          educ_2.0    1.007724
5          educ_3.0    3.489683
6          educ_4.0    3.276218
7          educ_5.0    2.978121
8          educ_6.0    5.310972
9          educ_7.0    2.268920
10         educ_8.0    3.067803
11        educ_10.0    2.933594
12        educ_11.0    1.388040
13        educ_12.0    1.007938
14      estrsoc_2.0    3.060427
15      estrsoc_3.0    6.362374
16      estrsoc_4.0   11.263736
17      estrsoc_5.0   15.844920
18      estrsoc_6.0   18.235744

--- Prueba de White de Heterocedasticidad ---
Estadístico LM: 118.5814
p-value LM: 0.0991

--- SELECCIÓN DE MODELOS (Criterios AIC y BIC) ---
Regla: Valores más bajos indican un mejor modelo.
Modelo 1 (Simple)   -> AIC: 12366.63  |  BIC: 12379.68
Modelo 2 (Múltiple) -> AIC: 12336.32  |  BIC: 12460.32


Interpretación:
En primer lugar, en cuanto la selección de modelos, al comparar ambos modelos, se observan resultados mixtos. El Modelo 2 (Múltiple) presenta un AIC de 12336.32, el cual es menor que el del Modelo 1 (12366.63), lo que indica un mejor ajuste. Sin embargo, su BIC es de 12460.32, que es mayor al del Modelo 1 (12379.68). Esta discrepancia ocurre porque el BIC penaliza de forma mucho más estricta la inclusión de muchas variables explicativas (todas las dummies de educación y estrato). A pesar de esto, elegimos el Modelo 2 como el principal, ya que el AIC respalda su ajuste y teóricamente necesitamos controlar por estas variables socioeconómicas para aislar el efecto real de la lengua indígena y evitar sesgos por variable omitida.
Asimismo, en cuanto a la variable lengua_indigena, nos enfocamos en el coeficiente de regresión parcial para nuestra variable de tratamiento, observando el resultado del Modelo 2. Manteniendo constantes todos los demás factores (edad, el sexo, el nivel educativo y el estrato social), tener como lengua materna una lengua indígena está asociado con una reducción promedio de 0.2256 puntos en el nivel de confianza hacia la Policía nacional. Al observar su valor p (P>|t| = 0.000), notamos que es netamente menor a 0.05, lo que significa que este efecto negativo es estadísticamente muy significativo.

Finalmente, en cuanto al VIF, nuestra variable de tratamiento (lengua_indigena = 1.15) y la mayoría de variables de control están por debajo del valor crítico de 10, lo que sugiere que no tenemos problemas de colinealidad con nuestra variable de interés. Sin embargo, las dummies de estrato social alto (estrsoc_4.0, estrsoc_5.0 y estrsoc_6.0) presentan VIFs superiores a 10 (llegando hasta 18.2). Esto indica que existe multicolinealidad entre los niveles de estratificación social, lo cual es esperable en este tipo de encuestas, pero afortunadamente no sesga ni afecta la precisión del coeficiente de nuestra variable principal (lengua indígena).
Por otro lado, en la prueba de White, obtuvimos un valor p de 0.0991. Al ser mayor a 0.05, no rechazamos la hipótesis nula de homocedasticidad. Esto es un excelente resultado, ya que indica que la varianza de los errores es constante en el modelo. Al no detectar un problema significativo de heterocedasticidad, los errores estándar clásicos reportados por nuestro modelo MCO son válidos y confiables.


### Modelos de Respuesta Binaria (Logit/Probit)

En esta sección, implementaremos los modelos Logit y Probit para estimar el efecto de hablar una lengua indígena sobre la confianza en la Policía Nacional.

In [30]:
import statsmodels.api as sm
import pandas as pd

# Definición de variables
# Y: Nivel de confianza en la Policía (1: Confianza superior a la mediana, 0: Poca/Ninguna o por debajo)
Y = data['conf_policia_dic']

# X: Variables independientes y de control (Alineadas con el MODELO MÚLTIPLE de la Persona 2)
# Variable principal: lengua_indigena
# Controles: edad, mujer, controles_educ, controles_estrato (Excluimos dominios regionales según el modelo MCO)
controles_educ = ['educ_2.0', 'educ_3.0', 'educ_4.0', 'educ_5.0', 'educ_6.0', 'educ_7.0', 'educ_8.0', 'educ_10.0', 'educ_11.0', 'educ_12.0']
controles_estrato = ['estrsoc_2.0', 'estrsoc_3.0', 'estrsoc_4.0', 'estrsoc_5.0', 'estrsoc_6.0']

cols_control = ['edad', 'mujer'] + controles_educ + controles_estrato

X = data[['lengua_indigena'] + cols_control]
X = sm.add_constant(X)


In [31]:
# Ajuste del modelo Logit
logit_model = sm.Logit(Y, X).fit()
print(logit_model.summary())


         Current function value: 0.482379
         Iterations: 35
                           Logit Regression Results                           
Dep. Variable:       conf_policia_dic   No. Observations:                 5046
Model:                          Logit   Df Residuals:                     5027
Method:                           MLE   Df Model:                           18
Date:                Mon, 02 Mar 2026   Pseudo R-squ.:                 0.02066
Time:                        02:26:08   Log-Likelihood:                -2434.1
converged:                      False   LL-Null:                       -2485.4
Covariance Type:            nonrobust   LLR p-value:                 7.032e-14
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
const              -1.5270      0.412     -3.706      0.000      -2.335      -0.719
lengua_indigena    -0.7505      0.098     -7.683  

/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [32]:
# Efectos Marginales para el modelo Logit
# Calculamos el AME (Average Marginal Effect)
logit_margeff = logit_model.get_margeff(method="dydx", at="overall")
print(logit_margeff.summary())


        Logit Marginal Effects       
Dep. Variable:       conf_policia_dic
Method:                          dydx
At:                           overall
                     dy/dx    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
lengua_indigena    -0.1153      0.015     -7.746      0.000      -0.144      -0.086
edad               -0.0007      0.000     -1.887      0.059      -0.001    2.75e-05
mujer               0.0066      0.011      0.577      0.564      -0.016       0.029
educ_2.0           -3.2218   1.17e+04     -0.000      1.000    -2.3e+04    2.29e+04
educ_3.0           -0.0065      0.030     -0.214      0.831      -0.066       0.053
educ_4.0            0.0257      0.031      0.832      0.406      -0.035       0.086
educ_5.0            0.0094      0.033      0.287      0.774      -0.055       0.074
educ_6.0            0.0274      0.031      0.891      0.373      -0.033       0.088
educ_7.0

#### Interpretación Logit (Efectos Marginales)

El cálculo de los Average Marginal Effects nos indica que hablar una lengua indígena disminuye la probabilidad de tener niveles altos de confianza en la Policía Nacional en un 11.53%, manteniendo todas las demás variables constantes (edad, sexo, educación y estrato socioeconómico). Este hallazgo es altamente estadísticamente significativo al 1% (p < 0.01).

Esta penalización de ~11.5% en la probabilidad de confiar en la autoridad puede explicarse por múltiples factores estructurales en el Perú:
1. La Policía Nacional del Perú, en gran parte de sus dependencias, no cuenta con personal bilingüe o un enfoque intercultural robusto. Esto dificulta que los ciudadanos con lengua materna indígena (quechua, aimara, lenguas amazónicas) puedan realizar denuncias o pedir ayuda en su propio idioma, generando una experiencia de exclusión institucional y desconfianza.
2. Históricamente, las poblaciones indígenas han sido grupos vulnerables frente a abusos de autoridad, discriminación estructural, y en muchos casos, han habitado en zonas con menor presencia protectora del Estado y mayor represión. Esto se traduce en una percepción de la policía como una entidad distante o punitiva en lugar de protectora.
3. Aunque el modelo controla por estrato socioeconómico y educación, hablar una lengua indígena suele intersectar con ubicaciones geográficas donde los servicios del Estado, incluyendo comisarías, son precarios, lentos o ineficaces, lo que merma la confianza institucional.

#### Modelo Probit

In [33]:
# Ajuste del modelo Probit
probit_model = sm.Probit(Y, X).fit()
print(probit_model.summary())


         Current function value: 0.482342
         Iterations: 35
                          Probit Regression Results                           
Dep. Variable:       conf_policia_dic   No. Observations:                 5046
Model:                         Probit   Df Residuals:                     5027
Method:                           MLE   Df Model:                           18
Date:                Mon, 02 Mar 2026   Pseudo R-squ.:                 0.02074
Time:                        02:26:09   Log-Likelihood:                -2433.9
converged:                      False   LL-Null:                       -2485.4
Covariance Type:            nonrobust   LLR p-value:                 6.003e-14
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
const              -0.9251      0.232     -3.983      0.000      -1.380      -0.470
lengua_indigena    -0.4188      0.053     -7.902  

/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [34]:
# Efectos Marginales para el modelo Probit
probit_margeff = probit_model.get_margeff(method="dydx", at="overall")
print(probit_margeff.summary())


       Probit Marginal Effects       
Dep. Variable:       conf_policia_dic
Method:                          dydx
At:                           overall
                     dy/dx    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
lengua_indigena    -0.1128      0.014     -7.986      0.000      -0.141      -0.085
edad               -0.0007      0.000     -1.873      0.061      -0.001    3.24e-05
mujer               0.0058      0.011      0.513      0.608      -0.016       0.028
educ_2.0           -1.5184   1.81e+04   -8.4e-05      1.000   -3.54e+04    3.54e+04
educ_3.0           -0.0061      0.029     -0.208      0.835      -0.064       0.051
educ_4.0            0.0279      0.030      0.932      0.351      -0.031       0.087
educ_5.0            0.0099      0.032      0.311      0.756      -0.053       0.072
educ_6.0            0.0276      0.030      0.920      0.357      -0.031       0.086
educ_7.0

#### Interpretación Probit (Efectos Marginales)

De manera casi idéntica al modelo Logit, los resultados marginales del modelo Probit indican que hablar una lengua indígena reduce la probabilidad de tener alta confianza en la policía en un 11.28%. Este resultado también es estadísticamente significativo al 1% (p < 0.01).

La diferencia entre el -11.53% del Logit y el -11.28% del Probit es minúscula (apenas 0.25 puntos porcentuales). Esto ocurre porque ambos asumen distribuciones probabilísticas en forma de "S" (Logística vs. Normal Estándar) que son empíricamente indistinguibles en el centro de la distribución. La consistencia entre ambos modelos nos da robustez: sin importar el supuesto matemático exacto que usemos, la conclusión es contundente y el impacto negativo es real.

#### Comparación entre el Modelo MCO y los Modelos Logit/Probit

Tanto el modelo de Mínimos Cuadrados Ordinarios (MCO) como los Modelos de Respuesta Binaria (Logit/Probit) llegan a la misma conclusión de fondo. Existe un efecto negativo y altamente significativo entre tener una lengua indígena y la confianza en la policía. Sin embargo, la forma en que miden este efecto es metodológicamente distinta.

1. Diferencia en la Variable Dependiente (Y):
*   El modelo MCO evaluó la confianza en su escala original del 1 al 4. El resultado de su regresión múltiple fue un coeficiente de -0.2256. Esto se interpreta como que hablar lengua indígena reduce el puntaje de confianza en aproximadamente un cuarto de punto (0.22) en esa escala, asumiendo saltos lineales entre "Nada" y "Poco", "Poco" y "Suficiente", etc.
*   Los modelos Logit y Probit Evalúan una variable binaria. El resultado transformado a Efectos Marginales arrojó un -11.5%. Esto se interpreta directamente en términos de probabilidades de ocurrencia: es un 11.5% menos probable que confíes en la policía si tu lengua materna es indígena.

Conclusión:
La similitud conceptual entre ambos frentes de trabajo demuestra una fuerte robustez estadística en el trabajo de investigación del equipo. La etnicidad/lengua materna es un predictor sociodemográfico claro de desconfianza institucional en el Perú.